# SYNTHETIC TECHNICAL VALIDATION — NOT BAF RESULTS
960 generated rows; reduced budgets; two CPU seeds. This checks execution and evidence, not research performance.

# 13 — Learning to Reweight và Meta-Weight-Net

Chạy LRE và MWN với meta holdout cân bằng nằm bên trong train. Không dùng tháng6/7 để cập nhật meta. Cần suite losses để có CE baseline cùng MLP. Mã được port sang torch.func vì nguồn LRE là TensorFlow cũ.

Không sửa cấu hình ở notebook này. Muốn đổi cấu hình hãy tạo campaign mới ở11.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = ROOT.parent.parent / ".cache" / "baf-research"
if os.name == "nt":
    CACHE_ROOT = Path(r"D:\UIT\.cache\baf-research")
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Project: /workspace/scratch/a3538503433e/work/research11/synthetic_validation
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05
Mode: synthetic
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317]


## 1. Chạy training có notebook output thật

        Cell khởi chạy một Python process mới bằng đúng interpreter của kernel. Process thực thi từng cell trong notebook con, lưu stdout/display/error sau từng cell và in log trực tiếp bên dưới. Fit hoàn tất được kiểm hash rồi reuse khi resume. Nếu bị ngắt, không xóa campaign. Cell này có thể chạy lâu.

In [3]:
SUITE = 'meta'
RUN_DIR = None
if SUITE in PLAN["suites"]:
    RUN_DIR = gru.research_launch_suite(CAMPAIGN, SUITE)
else:
    print("SKIPPED: suite không thuộc phạm vi đã chọn ở11:", SUITE)


BEGIN_STAGE 1
END_STAGE 1
BEGIN_STAGE 2
RESEARCH_STUDY meta mode= synthetic device= cpu
END_STAGE 2
BEGIN_STAGE 3
FIT_BEGIN lre_c00_s42 {'method': 'lre', 'lr': 0.03, 'candidate': 0, 'family': 'nn', 'normed': False, 'feature': 'dense'}
EPOCH lre 42 {'epoch': 1, 'loss': 0.7278295400437345, 'validation_ap': 0.15318873803606053, 'seconds': 0.043546069000512944}
EPOCH lre 42 {'epoch': 2, 'loss': 0.542535891358772, 'validation_ap': 0.13886991334515253, 'seconds': 0.039727731000311906}
FIT_COMPLETE lre_c00_s42 validation_AP= 0.15318873803606053
FIT_BEGIN mwn_c00_s42 {'method': 'mwn', 'lr': 0.03, 'candidate': 0, 'family': 'nn', 'normed': False, 'feature': 'dense'}
EPOCH mwn 42 {'epoch': 1, 'loss': 0.5700678329789237, 'validation_ap': 0.1749309003106517, 'seconds': 0.05887185200026579}
EPOCH mwn 42 {'epoch': 2, 'loss': 0.4647384707847338, 'validation_ap': 0.24832232435721413, 'seconds': 0.0682519729998603}
FIT_COMPLETE mwn_c00_s42 validation_AP= 0.24832232435721413
FIT_REUSED lre_c00_s42
FIT_BE

## 2. Xem kết quả validation đã khóa

        Chưa có test metric ở notebook này. Model, configuration và threshold được lưu trước khi mở test ở16.

In [4]:
if RUN_DIR is not None:
    display(gru.research_suite_overview(RUN_DIR))
    print("Notebook training thật:")
    for executed in sorted((RUN_DIR / "attempts").glob("*/training_executed.ipynb")):
        print(executed)


Notebook training thật:
/workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05/suites/meta/attempts/20261028_142653_d69fd169/training_executed.ipynb


,method,seed,validation_ap,threshold,fit_seconds,family,feature
0,lre,42,0.153189,inf,1.398886,nn,dense
1,lre,317,0.212252,0.749140,0.313589,nn,dense
2,mwn,42,0.248322,0.205953,0.369468,nn,dense
3,mwn,317,0.294405,0.123869,0.311027,nn,dense


## 3. Xuất proof trong một lượt

        Không cần Ctrl+S hoặc chạy cell cuối hai lần. Full proof giữ trên máy; review nhỏ hơn để gửi, ghi rõ phần đã bỏ. Notebook `verify.ipynb` được execute thật, kiểm hash/score và không fit lại.

In [5]:
if RUN_DIR is not None:
    PROOF = gru.research_export_proof(RUN_DIR, ROOT)
    for kind, path in PROOF.items():
        print(kind, "→", path)


RESEARCH_FROZEN_SUITE_VERIFIED 4
FULL_PROOF /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/meta_9fa6bed6_full_proof.zip
PARTIAL_REVIEW /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/meta_9fa6bed6_review.zip
full_proof → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/meta_9fa6bed6_full_proof.zip
review → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/meta_9fa6bed6_review.zip
